In [7]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import openassetpricing as oap

# baseline sample from Cotturo, Liu, and Proner (2025), "Multi-Factor Timing with Deep Learning"
SAMPLE_START = '1965-01-01'
SAMPLE_END = '2021-12-31'
OOS_CUTOFF = '1989-12-31'  # predictors missing any observation on/before this date are dropped (paper's OOS period starts 1990)

In [ ]:
# 5 factors: Mkt-RF, SMB, HML, RMW, CMA, RF
ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start=SAMPLE_START)[0]

# Momentum factor (separate file)
mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start=SAMPLE_START)[0]
mom.columns = mom.columns.str.strip()  # this file's column name often has extra whitespace

factors = ff5.join(mom, how='inner')
factors = factors / 100  # convert from percent to decimal — easy to forget
factors.index = factors.index.to_timestamp()  # PeriodIndex -> DatetimeIndex

factors = factors.loc[SAMPLE_START:SAMPLE_END]

# paper's 5 response factors (Mkt-RF is deliberately excluded so results aren't driven by market timing)
response_factors = factors[['SMB', 'HML', 'RMW', 'CMA', 'Mom']]

# fred-md macro data — raw pull only (no cleaning here)
# this is the *current* release; McCracken and Ng revise/update it monthly, so it reflects
# revisions made since the paper's data as of writing — see the cleaning cell for how this is handled
fredmd_url = "https://www.stlouisfed.org/-/media/project/frbstl/stlouisfed/research/fred-md/monthly/current.csv"
fredmd_raw = pd.read_csv(fredmd_url)

# financial predictors — raw pull only (no cleaning here)
# 'op' is openassetpricing's decile + long-short anomaly portfolio return dataset — this is what
# the paper actually uses ("long–short anomaly portfolio returns"), not firm-level characteristics
openap = oap.OpenAP(2023)  # August 2023 release ("Version 1.30"); this package version keys it by year, not YYYYMM
predictors_raw = openap.dl_port('op', 'pandas')

/var/folders/gh/dn12f24577v7qm098b5qxv580000gn/T/ipykernel_42857/224245314.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start=SAMPLE_START)[0]
/var/folders/gh/dn12f24577v7qm098b5qxv580000gn/T/ipykernel_42857/224245314.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start=SAMPLE_START)[0]
/var/folders/gh/dn12f24577v7qm098b5qxv580000gn/T/ipykernel_42857/224245314.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype


Data is downloaded: 4s


In [ ]:
# fred-md cleaning, following McCracken and Ng (2016) as described in the paper:
#   1. apply each series' McCracken-Ng transformation code to induce stationarity
#   2. lag by one additional month (t-1) to account for data announcement/release delays
#   3. drop any predictor with a missing value on/before OOS_CUTOFF — the paper's filter,
#      which keeps 122 of the raw file's 126 series
# remaining missing values (after 1990) are left as NaN here and imputed with the *expanding*
# training-set mean inside the walk-forward estimation loop, since imputing them globally now
# would leak future information into the training folds.

fredmd_tcodes = fredmd_raw.iloc[0, 1:].astype(int)
fredmd = fredmd_raw.iloc[1:].copy()
fredmd['sasdate'] = pd.to_datetime(fredmd['sasdate'])
fredmd = fredmd.set_index('sasdate').astype(float)


def _apply_fredmd_tcode(series, code):
    if code == 1:
        return series
    if code == 2:
        return series.diff()
    if code == 3:
        return series.diff().diff()
    if code == 4:
        return np.log(series)
    if code == 5:
        return np.log(series).diff()
    if code == 6:
        return np.log(series).diff().diff()
    if code == 7:
        return series.pct_change().diff()
    raise ValueError(f'unknown FRED-MD transform code: {code}')


fredmd_stationary = pd.DataFrame(
    {col: _apply_fredmd_tcode(fredmd[col], fredmd_tcodes[col]) for col in fredmd.columns}
)
fredmd_stationary = fredmd_stationary.shift(1)
fredmd_stationary.index = fredmd_stationary.index + pd.offsets.MonthEnd(0)  # align to month-end, like the predictors below
fredmd_stationary = fredmd_stationary.loc[SAMPLE_START:SAMPLE_END]

fredmd_keep = fredmd_stationary.loc[:OOS_CUTOFF].notna().all()

macro_predictors = fredmd_stationary.loc[:, fredmd_keep]

macro predictors: kept 122 of 126 (paper reports 122); dropped ['ACOGNO', 'ANDENOx', 'TWEXAFEGSMTHx', 'UMCSENTx']


In [ ]:
# financial predictor cleaning: keep only the 'LS' (long-short) leg of each anomaly's decile-sorted
# portfolio and reshape to one return column per anomaly.
ls_returns = predictors_raw[predictors_raw['port'] == 'LS'].copy()
ls_returns['date'] = pd.to_datetime(ls_returns['date']) + pd.offsets.MonthEnd(0)

financial = ls_returns.pivot(index='date', columns='signalname', values='ret') / 100  # percent -> decimal
financial = financial.loc[SAMPLE_START:SAMPLE_END]

# same missing-value filter as fred-md: drop anomalies missing any observation on/before OOS_CUTOFF,
# which keeps the paper's 137 of 212 available signals; remaining gaps are imputed later with the
# expanding training-set mean inside the estimation loop
financial_keep = financial.loc[:OOS_CUTOFF].notna().all()

financial_predictors = financial.loc[:, financial_keep]

In [11]:
financial_predictors.shape

(684, 137)

In [13]:
print(f"Your financial_predictors count: {financial_predictors.shape[1]} (paper reports 137)")
print(financial_predictors.columns.tolist())

Your financial_predictors count: 137 (paper reports 137)
['AM', 'Accruals', 'AdExp', 'AssetGrowth', 'BM', 'BMdec', 'BPEBM', 'Beta', 'BetaFP', 'BetaTailRisk', 'BidAskSpread', 'BookLeverage', 'CBOperProf', 'CF', 'CashProd', 'ChAssetTurnover', 'ChEQ', 'ChInv', 'ChInvIA', 'ChNNCOA', 'ChNWC', 'ChTax', 'CompEquIss', 'CompositeDebtIssuance', 'ConvDebt', 'CoskewACX', 'Coskewness', 'DelCOA', 'DelCOL', 'DelEqu', 'DelFINL', 'DelLTI', 'DelNetFin', 'DivInit', 'DivOmit', 'DivSeason', 'DivYieldST', 'DolVol', 'EBM', 'EP', 'EarnSupBig', 'EarningsConsistency', 'EarningsSurprise', 'EntMult', 'EquityDuration', 'ExchSwitch', 'FirmAge', 'Frontier', 'GP', 'GrLTNOA', 'GrSaleToGrInv', 'GrSaleToGrOverhead', 'Herf', 'HerfAsset', 'HerfBE', 'High52', 'IdioVol3F', 'IdioVolAHT', 'Illiquidity', 'IndMom', 'IndRetBig', 'IntMom', 'IntanCFP', 'IntanEP', 'IntanSP', 'InvGrowth', 'InvestPPEInv', 'Investment', 'LRreversal', 'Leverage', 'MRreversal', 'MaxRet', 'MeanRankRevGrowth', 'Mom12m', 'Mom12mOffSeason', 'Mom6m', 'MomOff